## learn conditional distribution of trimmed mean given trimmed SD

In [1]:
using Random, Statistics, Distributions
using Convex, MosekTools
using LinearAlgebra

# ============================================================
# 1) binning in U with merged tails
# ============================================================

"""
Make edges for U with merged tails:
- total bins = nbins_total
- left tail mass ~ q_bounds[1]
- right tail mass ~ 1-q_bounds[2]
- interior bins are quantile-equal between q_bounds[1] and q_bounds[2]
"""
function make_U_edges_merged_tails(U::Vector{Float64};
    nbins_total::Int = 60,
    q_bounds::Tuple{Float64,Float64} = (0.01, 0.99),
    jitter::Float64 = 0.0,
    rng::AbstractRNG = Random.default_rng(),
)
    @assert nbins_total >= 3 "need at least 3 bins (left tail, interior, right tail)"
    qlo, qhi = q_bounds
    @assert 0 < qlo < qhi < 1

    Uuse = jitter > 0 ? (U .+ jitter .* randn(rng, length(U))) : U

    interior_bins = nbins_total - 2
    probs = collect(range(qlo, qhi; length=interior_bins+1))   # includes qlo, qhi
    qs = quantile(Uuse, probs)                                 # length interior_bins+1
    cuts = qs[2:end-1]                                         # exclude endpoints

    # enforce strictly increasing cuts (handle ties)
    cuts = unique(cuts)
    if length(cuts) < interior_bins-1
        @warn "Many tied quantiles: interior bins reduced from $interior_bins to $(length(cuts)+1)"
    end

    edges = vcat(-Inf, cuts, Inf)
    return edges
end

"""
Assign each U[i] to a bin in 1:nbins (edges length = nbins+1).
"""
function bin_ids(U::Vector{Float64}, edges::Vector{Float64})
    nbins = length(edges) - 1
    b = searchsortedlast.(Ref(edges), U)
    return clamp.(b, 1, nbins)
end

function counts_by_edges(U::Vector{Float64}, edges::Vector{Float64})
    nbins = length(edges)-1
    b = searchsortedlast.(Ref(edges), U)
    b = clamp.(b, 1, nbins)
    counts = zeros(Int, nbins)
    @inbounds for bi in b
        counts[bi] += 1
    end
    return counts
end

# ============================================================
# 2) beta grid for mixture components
# ============================================================

"""
Default grid for mixture on beta.
"""
function default_grid_for_beta(β::Vector{Float64}; n_mu=25, n_sigma=40)
    qlo, qhi = quantile(β, (0.001, 0.999))
    pad = 0.2 * (qhi - qlo)
    mus = collect(range(qlo - pad, qhi + pad; length=n_mu))

    sβ = std(β)
    σ_lo = max(1e-3, 0.05 * sβ)
    σ_hi = max(σ_lo * 2, 2.0 * sβ)
    sigmas = exp.(range(log(σ_lo), log(σ_hi); length=n_sigma))

    return mus, sigmas
end

# ============================================================
# 6) h_hat: compute P(|β| >= t | U-bin) using fitted mixture
# ============================================================

@inline function two_sided_tail(d::UnivariateDistribution, t::Float64)
    tt = abs(t)
    return ccdf(d, tt) + cdf(d, -tt)
end

"""
Given fitted per-bin conditional mixture, compute h(t,v)
where v is D (not log D).
"""
function h_hat(fit, t::Float64, v::Float64)
    u = log(v + 1e-12)
    edges = fit.edges
    nbins = length(edges) - 1
    b = clamp(searchsortedlast(edges, u), 1, nbins)

    π = fit.π_bins[b]
    isempty(π) && return NaN

    meta = fit.comps
    s = 0.0
    @inbounds for k in 1:length(π)
        dk = component_dist(meta, k)
        s += π[k] * two_sided_tail(dk, t)
    end
    return s
end

h_hat

In [ ]:
th

In [2]:

# ============================================================
# 3) mixed grid: Normal + Student-t(ν) components
# ============================================================


function build_mixed_grid_separate(
    musN::Vector{Float64}, sigmasN::Vector{Float64};
    musT::Vector{Float64}=musN, sigmasT::Vector{Float64}=sigmasN,
    dofs::Vector{Int}=Int[3,5,8],
    include_normals::Bool=true,
    include_t::Bool=true
)
    mu_list   = Float64[]
    sig_list  = Float64[]
    fam_list  = Symbol[]
    dof_list  = Int[]

    if include_normals
        for μ in musN, σ in sigmasN
            push!(mu_list, μ); push!(sig_list, σ)
            push!(fam_list, :normal); push!(dof_list, 0)
        end
    end

    if include_t
        for ν in dofs, μ in musT, σ in sigmasT
            push!(mu_list, μ); push!(sig_list, σ)
            push!(fam_list, :t); push!(dof_list, ν)
        end
    end

    return (mu=mu_list, sigma=sig_list, family=fam_list, dof=dof_list)
end

@inline function component_dist(meta, k::Int)
    μ = meta.mu[k]
    σ = meta.sigma[k]
    if meta.family[k] === :normal
        return Normal(μ, σ)
    else
        ν = meta.dof[k]
        return LocationScale(μ, σ, TDist(ν))
    end
end

# ============================================================
# 4) fit mixture weights with MOSEK/Convex on fixed grid
# ============================================================

# """
# Fit weights π on a fixed grid of components (Normal + t).
# Objective: maximize Σ_i w_i log( Σ_k π_k f_k(x_i) )
# using log-sum-exp stabilization.

function fit_fixedgrid_mixed_mixture_mosek_weighted(
    x::Vector{Float64},
    w::Vector{Float64};
    musN::Vector{Float64},
    sigmasN::Vector{Float64},
    musT::Vector{Float64}=musN,
    sigmasT::Vector{Float64}=sigmasN,
    dofs::Vector{Int} = Int[3,5,8],
    include_normals::Bool = true,
    include_t::Bool = true,
    verbose::Bool=false
)
    @assert length(x) == length(w)
    N = length(x)

    meta = build_mixed_grid_separate(
        musN, sigmasN;
        musT=musT, sigmasT=sigmasT,
        dofs=dofs,
        include_normals=include_normals,
        include_t=include_t
    )
    K = length(meta.mu)

    logφ = Matrix{Float64}(undef, N, K)
    @inbounds for k in 1:K
        dk = component_dist(meta, k)
        for i in 1:N
            logφ[i, k] = logpdf(dk, x[i])
        end
    end

    m = vec(maximum(logφ, dims=2))
    A = exp.(clamp.(logφ .- m, -745.0, 0.0))

    π = Variable(K)
    constraints = [π >= 0, sum(π) == 1]
    obj = dot(w, m) + sum(w .* log(A * π))

    problem = maximize(obj, constraints)
    solve!(problem, Mosek.Optimizer; silent=!verbose)

    π_hat = vec(evaluate(π))
    π_hat = max.(π_hat, 0.0)
    π_hat ./= sum(π_hat)

    return π_hat, meta
end

fit_fixedgrid_mixed_mixture_mosek_weighted (generic function with 1 method)

In [3]:
# ============================================================
# 2b) within-bin β-based keep probabilities / weights
# ============================================================

"""
Piecewise keep probability based on z = abs.(βb) within a bin.

- z >= q_beta_tail quantile  -> keep prob = 1
- z <  q_beta_tail quantile  -> keep prob = a_min_beta

Returns:
  aβ::Vector{Float64}, info::NamedTuple
"""
function beta_keep_prob_piecewise(
    βb::Vector{Float64};
    q_beta_tail::Float64 = 0.95,
    a_min_beta::Float64 = 0.10
)
    @assert 0 < q_beta_tail < 1
    @assert 0 < a_min_beta <= 1

    z = abs.(βb)
    zR = quantile(z, q_beta_tail)

    aβ = similar(z, Float64)
    @inbounds for i in eachindex(z)
        aβ[i] = (z[i] >= zR) ? 1.0 : a_min_beta
    end

    info = (
        rule = :piecewise,
        q_beta_tail = q_beta_tail,
        zR = zR,
        a_min_beta = a_min_beta,
        frac_tail = mean(z .>= zR),
        mean_keep_prob = mean(aβ)
    )
    return aβ, info
end


"""
Apply β-based weighting or subsampling inside a bin.

Modes:
- beta_keep_rule = :none       -> returns original βb and all-ones weights
- beta_keep_rule = :piecewise  -> uses beta_keep_prob_piecewise

If beta_use_subsample=true:
  randomly keep points with prob aβ and assign IPW weights = 1/aβ on kept points
Else:
  keep all points and assign weights proportional to 1/aβ (reweight-only)

Returns:
  β_use, w_use, info
"""
function prepare_within_bin_beta_weighting(
    βb::Vector{Float64};
    rng::AbstractRNG = Random.default_rng(),
    beta_keep_rule::Symbol = :none,
    q_beta_tail::Float64 = 0.95,
    a_min_beta::Float64 = 0.10,
    beta_use_subsample::Bool = false,
    rescale_beta_weights::Bool = true
)
    n = length(βb)
    n == 0 && return βb, Float64[], (rule=:none, n_in=0, n_used=0)

    # no weighting
    if beta_keep_rule == :none
        return βb, ones(Float64, n), (rule=:none, n_in=n, n_used=n, mean_weight=1.0, max_weight=1.0)
    end

    # get keep probs
    if beta_keep_rule == :piecewise
        aβ, info0 = beta_keep_prob_piecewise(βb; q_beta_tail=q_beta_tail, a_min_beta=a_min_beta)
    else
        error("Unknown beta_keep_rule=$beta_keep_rule. Supported: :none, :piecewise")
    end

    if beta_use_subsample
        keep = rand(rng, n) .< aβ
        idx = findall(keep)

        β_use = βb[idx]
        w_use = 1.0 ./ aβ[idx]

        if rescale_beta_weights && !isempty(w_use)
            # normalize so sum(weights) ≈ number used
            w_use .*= (length(w_use) / sum(w_use))
        end

        info = (; info0..., n_in=n, n_used=length(β_use),
                 used_frac=length(β_use)/n,
                 mean_weight=(isempty(w_use) ? NaN : mean(w_use)),
                 max_weight=(isempty(w_use) ? NaN : maximum(w_use)))
        return β_use, w_use, info
    else
        # weight-only, no subsampling
        β_use = βb
        w_use = 1.0 ./ aβ

        if rescale_beta_weights && !isempty(w_use)
            w_use .*= (length(w_use) / sum(w_use))
        end

        info = (; info0..., n_in=n, n_used=n, used_frac=1.0,
                 mean_weight=mean(w_use), max_weight=maximum(w_use))
        return β_use, w_use, info
    end
end

prepare_within_bin_beta_weighting

In [4]:
function fit_conditional_beta_given_U_bins(
    β::Vector{Float64},
    U::Vector{Float64};
    nbins_total::Int = 60,
    q_bounds::Tuple{Float64,Float64} = (0.01,0.99),
    jitter::Float64 = 1e-10,
    rng::AbstractRNG = Random.default_rng(),

    # separate grid sizes
    n_mu_N::Int = 25,
    n_sigma_N::Int = 40,
    n_mu_T::Int = 9,
    n_sigma_T::Int = 15,

    dofs::Vector{Int} = Int[3,5,8,12,20],
    include_normals::Bool = true,
    include_t::Bool = true,
    min_bin_size::Int = 500,
    verbose::Bool = false,
    bins_to_fit::Union{Nothing, Vector{Int}} = nothing,

    # optional: widen t sigmas upward for tail focus
    widen_t_sigma_mult::Float64 = 1.0,

    # NEW: within-bin β-based weighting/subsampling
    beta_keep_rule::Symbol = :none,       # :none or :piecewise
    q_beta_tail::Float64 = 0.95,          # top 5% |β| treated as tail
    a_min_beta::Float64 = 0.10,           # center keep prob
    beta_use_subsample::Bool = false,     # false = weight-only; true = actual subsample+IPW
    rescale_beta_weights::Bool = true
)
    @assert length(β) == length(U)

    edges = make_U_edges_merged_tails(U; nbins_total=nbins_total,
                                      q_bounds=q_bounds, jitter=jitter, rng=rng)

    nbins = length(edges) - 1
    b_id = bin_ids(U, edges)

    # shared grids across bins
    musN, sigmasN = default_grid_for_beta(β; n_mu=n_mu_N, n_sigma=n_sigma_N)
    musT, sigmasT = default_grid_for_beta(β; n_mu=n_mu_T, n_sigma=n_sigma_T)

    if widen_t_sigma_mult != 1.0
        sigmasT = exp.(range(log(minimum(sigmasT)),
                             log(maximum(sigmasT) * widen_t_sigma_mult);
                             length=length(sigmasT)))
    end

    π_bins = [Float64[] for _ in 1:nbins]
    counts = zeros(Int, nbins)              # original count in each bin
    counts_used = zeros(Int, nbins)         # actual rows used in fit (after β subsampling if enabled)
    weight_sums = zeros(Float64, nbins)     # sum of weights used in fit
    comps_ref = nothing

    # store diagnostics per bin
    # beta_weight_stats = [nothing for _ in 1:nbins]
    beta_weight_stats = Vector{Any}(undef, nbins)
    fill!(beta_weight_stats, nothing)

    fit_set = bins_to_fit === nothing ? collect(1:nbins) : unique(clamp.(bins_to_fit, 1, nbins))
    fit_flag = falses(nbins)
    fit_flag[fit_set] .= true

    for b in 1:nbins
        idx = findall(==(b), b_id)
        counts[b] = length(idx)

        if !fit_flag[b] || counts[b] < min_bin_size
            continue
        end

        βb = β[idx]

        # NEW: within-bin β-based weighting / subsampling
        β_use, wb, infoβ = prepare_within_bin_beta_weighting(
            βb;
            rng=rng,
            beta_keep_rule=beta_keep_rule,
            q_beta_tail=q_beta_tail,
            a_min_beta=a_min_beta,
            beta_use_subsample=beta_use_subsample,
            rescale_beta_weights=rescale_beta_weights
        )

        beta_weight_stats[b] = infoβ
        counts_used[b] = length(β_use)
        weight_sums[b] = isempty(wb) ? 0.0 : sum(wb)

        # if after subsampling too few points remain, skip
        if counts_used[b] < min_bin_size
            continue
        end

        π_hat, meta = fit_fixedgrid_mixed_mixture_mosek_weighted(
            β_use, wb;
            musN=musN, sigmasN=sigmasN,
            musT=musT, sigmasT=sigmasT,
            dofs=dofs,
            include_normals=include_normals,
            include_t=include_t,
            verbose=verbose
        )

        π_bins[b] = π_hat
        comps_ref === nothing && (comps_ref = meta)
    end

    return (
        edges=edges, π_bins=π_bins, comps=comps_ref,
        counts=counts, counts_used=counts_used, weight_sums=weight_sums,
        bins_fit=fit_set,
        grids=(musN=musN, sigmasN=sigmasN, musT=musT, sigmasT=sigmasT),
        beta_weight_rule=beta_keep_rule,
        beta_use_subsample=beta_use_subsample,
        beta_weight_stats=beta_weight_stats
    )
end

fit_conditional_beta_given_U_bins (generic function with 1 method)

In [5]:
@inline function trim_drop1(x::Vector{Float64})
    n = length(x)
    @assert n ≥ 3 "Need n ≥ 3 to drop min and max"
    xs = sort(x)
    return @view xs[2:n-1]   # length = n-2
end

@inline function sd_unbiased(x)
    m = length(x)
    @assert m ≥ 2 "Need at least 2 points for SD"
    μ = mean(x)
    return sqrt(sum((x .- μ).^2) / (m - 1))
end

sd_unbiased (generic function with 1 method)

In [6]:
using Random, Statistics, Distributions

"""
Simulate B datasets of size n from N(0,1), return:
  β = sqrt(n) * mean(x_trim)
  S = sd_unbiased(x_trim)   (unbiased SD on trimmed sample; denom = (n-2)-1)
  U = log(S)
where x_trim drops min and max.
"""
function simulate_beta_Strim_U(n::Int, B::Int; rng=Random.default_rng())
    @assert n ≥ 4 "Need n ≥ 4 so trimmed sample has ≥2 points for SD"
    β = Vector{Float64}(undef, B)
    S = Vector{Float64}(undef, B)
    x = Vector{Float64}(undef, n)
    dist = Normal(0,1)

    @inbounds for b in 1:B
        rand!(rng, dist, x)
        xt = trim_drop1(x)              # length m = n-2

        mμ = mean(xt)
        s  = sd_unbiased(xt)

        β[b] = sqrt(n) * mμ
        S[b] = s
    end

    U = log.(S .+ 1e-300)
    return β, S, U
end

simulate_beta_Strim_U

## n=4

In [7]:
rng = MersenneTwister(3)
n = 4
B = 1_100_000_000

β, S_trim, U = simulate_beta_Strim_U(n, B; rng=rng)

fit_trimSD = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 65.65 seconds, 66.670 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240107 (720321 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241217          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 72.91 seconds, 59.605 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 219124 (657372 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 220234          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.41            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 79.16 seconds, 59.410 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218195 (654585 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219305          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.55            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 80.31 seconds, 59.413 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218377 (655131 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219487          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.56            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524387 bytes.

(edges = [-Inf, -5.137458532234867, -4.487519727679684, -4.094996747758447, -3.8122525021908196, -3.590571085958769, -3.4082715608074827, -3.253111715229239, -3.117931317160281, -2.998104424335508  …  -0.057776975467264964, -0.020829371825151308, 0.018734219029347802, 0.06159040200712951, 0.10883440764324213, 0.16222698316109008, 0.22482823114770634, 0.30301204890453765, 0.41464780144057134, Inf], π_bins = [[2.792600660016625e-7, 1.68916239737832e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.9656050360653312e-8  …  0.0, 0.0, 1.680062769225396e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.7039027668814763e-8, 1.3602245025818022e-10, 0.0, 0.0, 0.0, 0.0, 0.0, 4.6557694122451504e-8, 7.009484421435615e-8, 2.424960249161307e-8  …  0.0, 0.0, 3.829342647351484e-8, 3.4043946670691374e-8, 3.285102920741982e-8, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0417704756314467e-8, 7.940959110448736e-8, 3.24574881001328e-7  …  0.0, 0.0, 1.0634533027415526e-5, 0.0, 1.2234686876483092e-7, 1.94

In [8]:
using JLD2
@save "fit_beta_subsample_1.4_trimSD(n=4).jld2" fit_trimSD

In [9]:
fit_trimSD

(edges = [-Inf, -5.137458532234867, -4.487519727679684, -4.094996747758447, -3.8122525021908196, -3.590571085958769, -3.4082715608074827, -3.253111715229239, -3.117931317160281, -2.998104424335508  …  -0.057776975467264964, -0.020829371825151308, 0.018734219029347802, 0.06159040200712951, 0.10883440764324213, 0.16222698316109008, 0.22482823114770634, 0.30301204890453765, 0.41464780144057134, Inf], π_bins = [[2.792600660016625e-7, 1.68916239737832e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.9656050360653312e-8  …  0.0, 0.0, 1.680062769225396e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.7039027668814763e-8, 1.3602245025818022e-10, 0.0, 0.0, 0.0, 0.0, 0.0, 4.6557694122451504e-8, 7.009484421435615e-8, 2.424960249161307e-8  …  0.0, 0.0, 3.829342647351484e-8, 3.4043946670691374e-8, 3.285102920741982e-8, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0417704756314467e-8, 7.940959110448736e-8, 3.24574881001328e-7  …  0.0, 0.0, 1.0634533027415526e-5, 0.0, 1.2234686876483092e-7, 1.94

## n=5

In [7]:
rng = MersenneTwister(3)
n = 5
B = 1_100_000_000

β, S_trim, U = simulate_beta_Strim_U(n, B; rng=rng)

fit_trimSD = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 66.62 seconds, 66.930 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 241116 (723348 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 242226          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 75.54 seconds, 59.474 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218614 (655842 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219724          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 79.96 seconds, 59.389 GiB of memory allocated


...

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 82.22 seconds, 59.382 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218257 (654771 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219367          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.53            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 80.28 seconds, 59.275 GiB of memory allocated
Excessive output truncated after 524361 bytes.

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)


(edges = [-Inf, -2.9444987420329487, -2.6093226738583133, -2.404849987740245, -2.2564795244863123, -2.1395307818611498, -2.0426887467154162, -1.959884510493351, -1.8873909512044298, -1.8227993387697516  …  -0.05005890430632125, -0.023506745205454576, 0.00509750113911258, 0.03632103835727533, 0.07102132736432774, 0.11060747032971491, 0.15750212309119377, 0.21683534143037672, 0.30309954696429625, Inf], π_bins = [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.01266632406577e-7  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [7.453732881229943e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 1.5913797578899999e-9, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.2920506259171007e-7, 0.0, 2.662128006004402e-7, 6.820702429177593e-9, 0.0, 0.0  …  0.0, 0.0, 3.4956601251346534e-8, 5.8703784717234933e-8, 0.0, 0.0, 1.0218885084194166e-7, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 2.657563763600432e-8, 1.76617868114735e-6, 0.0, 1.0432317732042245e-7, 2.0099631190205268e-8,

In [8]:
using JLD2
@save "fit_beta_subsample_1.4_trimSD(n=5).jld2" fit_trimSD

## n=6

In [7]:
rng = MersenneTwister(3)
n = 6
B = 1_100_000_000

β, S_trim, U = simulate_beta_Strim_U(n, B; rng=rng)

fit_trimSD = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 65.77 seconds, 66.623 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 239925 (719775 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241035          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 72.39 seconds, 59.468 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218592 (655776 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219702          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 79.05 seconds, 59.391 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218230 (654690 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219340          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.57            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 76.73 seconds, 59.365 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218190 (654570 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219300          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.60            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524372 bytes.

(edges = [-Inf, -2.1805585496583837, -1.948299285738297, -1.8055805784332861, -1.7015344851661434, -1.6191057889244689, -1.5506384339421668, -1.491840009442076, -1.4401824539386194, -1.3940412019625885  …  -0.04337536960802624, -0.021336925170386538, 0.0024855735736215388, 0.028592715138470143, 0.05773667322746075, 0.09110388866237885, 0.13085241651924043, 0.18148386416803688, 0.2557460879326607, Inf], π_bins = [[0.0, 0.0, 0.0, 0.0, 0.0, 1.0244020602576688e-7, 2.8386004697950426e-6, 1.3969135062592068e-8, 0.0, 0.0  …  0.0, 3.2404342694839877e-9, 0.0, 0.0, 1.0586043122657533e-8, 3.3781320584598694e-8, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.858612652471107e-7, 0.0, 0.0  …  0.0, 7.462663791434822e-9, 0.0, 0.0, 0.0, 9.085993752035791e-7, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3389313007918463e-8, 2.3578005814683105e-7  …  0.0, 0.0, 0.0, 0.0, 0.0, 6.20183212615222e-8, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0374536053180882e-6, 1.2425934499292047e

In [9]:
using JLD2
@save "fit_beta_subsample_1.4_trimSD(n=6).jld2" fit_trimSD

In [8]:
fit_trimSD

(edges = [-Inf, -2.1805585496583837, -1.948299285738297, -1.8055805784332861, -1.7015344851661434, -1.6191057889244689, -1.5506384339421668, -1.491840009442076, -1.4401824539386194, -1.3940412019625885  …  -0.04337536960802624, -0.021336925170386538, 0.0024855735736215388, 0.028592715138470143, 0.05773667322746075, 0.09110388866237885, 0.13085241651924043, 0.18148386416803688, 0.2557460879326607, Inf], π_bins = [[0.0, 0.0, 0.0, 0.0, 0.0, 1.0244020602576688e-7, 2.8386004697950426e-6, 1.3969135062592068e-8, 0.0, 0.0  …  0.0, 3.2404342694839877e-9, 0.0, 0.0, 1.0586043122657533e-8, 3.3781320584598694e-8, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.858612652471107e-7, 0.0, 0.0  …  0.0, 7.462663791434822e-9, 0.0, 0.0, 0.0, 9.085993752035791e-7, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3389313007918463e-8, 2.3578005814683105e-7  …  0.0, 0.0, 0.0, 0.0, 0.0, 6.20183212615222e-8, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0374536053180882e-6, 1.2425934499292047e

## n= 8

In [7]:
rng = MersenneTwister(3)
n = 8
B = 1_100_000_000

β, S_trim, U = simulate_beta_Strim_U(n, B; rng=rng)

fit_trimSD = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 65.1 seconds, 66.712 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240269 (720807 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241379          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.43            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 81.01 seconds, 59.459 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218556 (655668 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219666          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.43            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 84.91 seconds, 59.681 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 219192 (657576 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 220302          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.60            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 76.43 seconds, 59.401 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218329 (654987 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219439          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.61            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524365 bytes.

(edges = [-Inf, -1.5268816256246092, -1.3768464345552573, -1.2837235141046575, -1.2152843285882375, -1.1607293950552746, -1.1151497990684631, -1.0758563432118877, -1.0412042665118855, -1.010134636843051  …  -0.02928535438588004, -0.011779047828894077, 0.007199111855020724, 0.028069340177981267, 0.05146512062655129, 0.078400810907871, 0.1106534390697039, 0.1520192745935074, 0.21326876075723344, Inf], π_bins = [[0.0, 3.6951946365499184e-10, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0036119084783928e-7  …  0.0, 0.0, 0.0, 0.0, 0.0, 7.744569268035896e-8, 2.0369845721067122e-7, 0.0, 0.0, 0.0], [2.7750539846642956e-6, 7.248762018746357e-7, 1.8096196729157094e-7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 7.115853572072056e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.615829995795011e-8, 5.770696683189942e-7  …  0.0, 1.4076537994016952e-7, 0.0, 0.0, 6.585098503555668e-8, 2.353364727413112e-7, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

In [10]:
using JLD2
@save "fit_beta_subsample_1.4_trimSD(n=8).jld2" fit_trimSD

In [9]:
fit_trimSD

(edges = [-Inf, -1.5268816256246092, -1.3768464345552573, -1.2837235141046575, -1.2152843285882375, -1.1607293950552746, -1.1151497990684631, -1.0758563432118877, -1.0412042665118855, -1.010134636843051  …  -0.02928535438588004, -0.011779047828894077, 0.007199111855020724, 0.028069340177981267, 0.05146512062655129, 0.078400810907871, 0.1106534390697039, 0.1520192745935074, 0.21326876075723344, Inf], π_bins = [[0.0, 3.6951946365499184e-10, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0036119084783928e-7  …  0.0, 0.0, 0.0, 0.0, 0.0, 7.744569268035896e-8, 2.0369845721067122e-7, 0.0, 0.0, 0.0], [2.7750539846642956e-6, 7.248762018746357e-7, 1.8096196729157094e-7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 7.115853572072056e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.615829995795011e-8, 5.770696683189942e-7  …  0.0, 1.4076537994016952e-7, 0.0, 0.0, 6.585098503555668e-8, 2.353364727413112e-7, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

## n=10

In [7]:
rng = MersenneTwister(3)
n = 10
B = 1_100_000_000

β, S_trim, U = simulate_beta_Strim_U(n, B; rng=rng)

fit_trimSD = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 70.52 seconds, 66.865 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240866 (722598 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241976          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.46            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 77.18 seconds, 59.425 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218423 (655269 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219533          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 80.5 seconds, 59.489 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218296 (654888 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219406          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.61            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 90.44 seconds, 59.368 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218201 (654603 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219311          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.59            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524338 bytes.

(edges = [-Inf, -1.220532214862716, -1.1058037732907442, -1.0341626888762794, -0.9812596389451507, -0.9389434117497163, -0.903466853754809, -0.8727878684896411, -0.8456524150218447, -0.8212764921160264  …  -0.017847043255553044, -0.002821062608933743, 0.013507779457662676, 0.031511848828975626, 0.05173710374939538, 0.07506057193067026, 0.10309050958268584, 0.13915484139990053, 0.19282960385512973, Inf], π_bins = [[0.0, 1.0659432986580706e-8, 0.0, 0.0, 0.0, 0.0, 1.1668675274491711e-9, 3.157298774514458e-7, 5.184668434638253e-8, 2.053494293553868e-6  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [9.752952890160115e-9, 0.0, 0.0, 0.0, 0.0, 1.9896080162450018e-8, 0.0, 0.0, 0.0, 3.8275251209776366e-8  …  0.0, 0.0, 0.0, 2.858147295188371e-7, 0.0, 0.0, 4.59226732063705e-8, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 2.081110288402049e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 1.1091746383482392e-8, 0.0, 0.0, 0.0, 0.0], [8.562887077059766e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.58637

In [8]:
using JLD2
@save "fit_beta_subsample_1.4_trimSD(n=10).jld2" fit_trimSD